# READ DATA

In [1]:
import pandas as pd
import duckdb
import os


# Define your tables
table_names = [
    'orders',
    'customers', 
    'products',
    'sellers',
    'order_payments',
    'order_items',
    'order_reviews',
    'geoloc'
]

# Load all into a dictionary
datasets = {
    name: pd.read_parquet(f'../data/outputs/{name}_raw.parquet')
    for name in table_names
}

# LOAD DATA TO DUCKDB

In [2]:
# Create a connection to SQLlite
# This will create .db file if it doesn't exist
conn = duckdb.connect()

# load a dataframe in into duckdb

def load_df_to_sql(dataframes: dict):
    conn = duckdb.connect('../database/olist.db')
    for nm, df in dataframes.items():
        # DuckDB sees the variable 'df' in your local Python scope automatically
        df = df.convert_dtypes()
        conn.execute(f"CREATE OR REPLACE TABLE {nm} AS SELECT * FROM df")
        print(f" Loaded: {nm}")
        # FIX THE VARCHARS: Run a quick alter for known date columns
        # This is much faster than doing it in Pandas

    cols = df.columns
    for col in cols:
        if 'timestamp' in col or 'at' in col or'date' in col:
            try:
                conn.execute(f"ALTER TABLE {nm} ALTER {col} TYPE TIMESTAMP")
            except:
                pass # Skip if it's not actually a date string
        
load_df_to_sql(datasets)


  

 Loaded: orders
 Loaded: customers
 Loaded: products
 Loaded: sellers
 Loaded: order_payments
 Loaded: order_items
 Loaded: order_reviews
 Loaded: geoloc


# ORDERS

In [ ]:
# reading data from orders


conn = duckdb.connect('../database/olist.db')
# Execute and fetch as list of tuples

# Convert the base columns to real timestamps one by one
date_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in date_cols:
    print(f"Optimizing {col}...")
    conn.execute(f"ALTER TABLE orders ALTER {col} TYPE TIMESTAMP USING TRY_CAST({col} AS TIMESTAMP)")

result = conn.execute("select * from orders").df().head()
print(result)  # [(2,)]


                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08

In [56]:
# calculating number of days till the approval, shipment, delivery

query = """ 
CREATE OR REPLACE TABLE orders_delivery_details AS 
SELECT 
    order_id, customer_id, order_status,
    datediff('day', order_purchase_timestamp, order_approved_at) as days_till_approval,
    datediff('day', order_purchase_timestamp, order_delivered_carrier_date ) as days_till_shipped,
    datediff('day', order_purchase_timestamp , order_delivered_customer_date ) as days_till_delivery,
    datediff('day', order_purchase_timestamp , order_estimated_delivery_date ) as days_till_estim_delivery,
    datediff('day', order_estimated_delivery_date, order_delivered_customer_date) AS delivery_accuracy
from orders """
conn.execute(query)


In [57]:
# reading the order delivery details
query = """
    SELECT * from orders_delivery_details
"""
df=conn.execute(query).df()
print(df)

                               order_id                       customer_id  \
0      e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1      53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2      47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3      949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4      ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   
...                                 ...                               ...   
99436  9c5dedf39a927c1b2549525ed64a053c  39bd1228ee8140590ac3aca26f2dfe00   
99437  63943bddc261676b46f01ca7ac2f7bd8  1fca14ff2861355f6e5f14306ff977a7   
99438  83c1379a015df1e13d02aae0204711ab  1aa71eb042121263aafbe80c1b562c9c   
99439  11c177c8e97725db2631073c19f07b62  b331b74b18dc79bcdf6532d51e1637c1   
99440  66dea50a8b16d9b4dee7af250b4be1a5  edb027a75a1449115f6b43211ae02a24   

      order_status  days_till_approval  days_till_shipped  days_till_delive

# PRODUCTS

In [7]:
# reading data from products

# Execute and fetch as list of tuples
result = conn.execute("select * from products").df().head()

# Convert the base columns to integers one by one
cols = [
    'product_name_lenght', 
    'product_description_lenght', 
    'product_photos_qty', 
    'product_weight_g', 
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

for col in cols:
    print(f"Optimizing {col}...")
    conn.execute(f"ALTER TABLE products ALTER {col} TYPE INTEGER USING TRY_CAST({col} AS INTEGER)")

Optimizing product_name_lenght...
Optimizing product_description_lenght...
Optimizing product_photos_qty...
Optimizing product_weight_g...
Optimizing product_length_cm...
Optimizing product_height_cm...
Optimizing product_width_cm...


# ORDER ITEMS

In [8]:
# reading data from products

# Convert the base columns to float one by one
cols = [
    'price',
    'freight_value'
]

for col in cols:
    print(f"Optimizing {col}...")
    conn.execute(f"ALTER TABLE order_items ALTER {col} TYPE FLOAT USING TRY_CAST({col} AS FLOAT)")

# Convert the base column to date columns
col = 'shipping_limit_date'
print(f"Optimizing {col}...")
conn.execute(f"ALTER TABLE order_items ALTER {col} TYPE TIMESTAMP USING TRY_CAST({col} AS TIMESTAMP)")

# Execute and fetch as list of tuples
result = conn.execute("select * from order_items").df().head()
print(result)


Optimizing price...
Optimizing freight_value...
Optimizing shipping_limit_date...
                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   
3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   
4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87   

  shipping_limit_date       price  freight_value  
0 2017-09-19 09:45:35   58.900002      13.290000  
1 2017-05-03 11:05:13  239.899994      19.930000  
2

In [ ]:
# aggregating the order items data to calculate total cost, freight value, shipping burden and total cost to customer
query = """
    CREATE OR REPLACE TABLE order_cost_detils AS 
    select 
        order_id, shipping_limit_date, 
        count(distinct product_id) as total_products, 
        count(order_item_id) as total_items, 
        round(sum(price),4) as total_cost, 
        round(sum(freight_value),4) as total_freight_value,
        round(sum(price+freight_value),4) as total_cost_customer,
        round(sum(freight_value/price),4) as shipping_burden
    from order_items
    group by order_id, shipping_limit_date       
"""
conn.execute(query)


In [ ]:
query = """
    SELECT * from order_cost_detils
"""
res = conn.execute(query).df().head()
print(result)